# IOAI — 2025 Selection Camp Toxic Comments (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/roai-toxic.zip', '/tmp/tox.zip')
    with zipfile.ZipFile('/tmp/tox.zip') as zf: zf.extractall('data')
print('데이터 준비:', 'data/train.csv' if os.path.exists('data/train.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Toxic Comments — 악성 댓글 멀티라벨 분류 (베이스라인)
댓글을 **toxic / severe_toxic / obscene / insult** 4개 이진 라벨로 분류합니다.
채점 = held-out(`int(id,16)%5==0`) **4라벨 F1 평균(mean_f1, 높을수록 좋음)**.

베이스라인: **단어 TF-IDF + 라벨별 로지스틱 회귀**(임계값 0.5). (개선: 단어+글자 TF-IDF + 임계값 튜닝 → 모범답안)

In [ ]:
import pandas as pd\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nLABELS = ['toxic','severe_toxic','obscene','insult']\ndf = pd.read_csv('data/train.csv'); df['comment_text']=df['comment_text'].fillna('')\nis_val = df['id'].apply(lambda s:int(str(s),16)%5==0)\ntr, va = df[~is_val].reset_index(drop=True), df[is_val].reset_index(drop=True)\nprint('train',len(tr),'val',len(va))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
vec = TfidfVectorizer(max_features=20000, ngram_range=(1,1))
Xtr = vec.fit_transform(tr['comment_text']); Xva = vec.transform(va['comment_text'])
pred = {}
for c in LABELS:
    clf = LogisticRegression(max_iter=300, class_weight='balanced').fit(Xtr, tr[c])
    pred[c] = clf.predict(Xva)
import numpy as np
print('mean F1:', round(np.mean([f1_score(va[c], pred[c], zero_division=0) for c in LABELS]),4))
out = pd.DataFrame({'id': va['id']});
for c in LABELS: out[c] = pred[c]
out.to_csv('submission.csv', index=False); print('wrote submission.csv', len(out))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)